# 06 Window noise, detector calibration and power

Self-contained: this notebook downloads NFCorpus and two embedding models from the Hugging Face Hub, embeds everything, and computes every number and figure from scratch. It does not read `results/`, `cache/`, or any `silent_rag` module, so it runs on a fresh machine with only the pip packages below. Expect roughly 10-20 minutes on a laptop CPU.

The benchmark scores **windows** of queries (100 by default), but per-query correlations do not say how a window-level detector behaves. This notebook answers the questions a detector benchmark needs first.

**Questions**
1. How noisy is window-mean nDCG@10 on clean data, and what is the false-alarm rate of the "harmful" rule (paired bootstrap, p < 0.05) at window size 100?
2. Are KS and z-score detectors calibrated against a fixed reference window?
3. Do window-level label-free signals track window nDCG (hypothesis H3), and how well do they separate failing queries?
4. How much damage is detectable, and how early? Is the harmful label itself stable?

**Two query populations (second-review change).** The runner (`src/silent_rag/run.py` with `data.load(split="test")`) sees only the **323 NFCorpus test queries**: a 100-query reference and a 223-query pool, from which every trial window of 100 is drawn. An earlier version of this notebook calibrated on all 3,237 labelled queries, which is a different population: a smaller pool shrinks the window SD (finite-population correction 0.74 instead of 0.98) and makes trial windows overlap heavily (about 45 shared queries between two windows instead of about 3). Every window-level section is therefore run on:
- **`runner` (primary):** the test queries, with the reference and pool chosen exactly as `run_dataset` does (`default_rng(seed).permutation`, first 100 = reference).
- **`all` (sensitivity):** all 3,237 labelled queries, as before.

**Approach**
Random clean windows against the finite-population variance formula; a sign-flip null for the harmful rule; false-alarm rates over 200 reference draws; window-level correlation with a permutation null; power and earliness under a synthetic degradation (a fraction of queries return random documents); a Monte-Carlo check of how often the harmful call depends on the bootstrap seed; minimum detectable effect.

**Not the project's failure injectors.** The synthetic degradation is a placeholder with a known effect size so the statistics can be checked. It is not F1-F5.

**Scope.** NFCorpus only, the project's development set. SciFact and FiQA are held out for the final run, so this notebook deliberately does not load them.

In [ ]:
# Uncomment on a fresh environment.
# %pip install -q datasets sentence-transformers scikit-learn scipy pandas numpy matplotlib

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # macOS: OpenMP clash between libraries; harmless elsewhere
os.environ.setdefault("OMP_NUM_THREADS", "1")

import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy import stats
from scipy.stats import ks_2samp
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 30)

SEED = 0
np.random.seed(SEED)
K = 10
W = 100                                     # window size
SPLITS = ["train", "validation", "test"]    # the Hub qrels name is "validation", not "dev"
MAX_SEQ = 512

NFCORPUS = dict(
    repo="BeIR/nfcorpus", revision="b5026a0e96e8a7ac4f95f482a596389289d46269",
    qrels="BeIR/nfcorpus-qrels", qrels_revision="a451b3b26d3ae1358f259c1a3a4dd61fcea35a65",
)
MODELS = {
    "multi-qa-MiniLM": dict(repo="sentence-transformers/multi-qa-MiniLM-L6-cos-v1", revision="b207367332321f8e44f96e224ef15bc607f4dbf0"),
    "all-MiniLM": dict(repo="sentence-transformers/all-MiniLM-L6-v2", revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41"),
}

## Helpers, data and clean retrieval

Data loading, encoding, exact cosine search, nDCG@10 (graded gain, log2 discount, as in BEIR), and the paired bootstrap used for the harmful rule: for per-query clean and damaged scores, the mean drop `clean - damaged` and a one-sided p-value for "no drop" (the share of bootstrap means <= 0). `paired_bootstrap` is a copy of `src/silent_rag/eval/ground_truth.py:paired_bootstrap` (same defaults, `n_boot = 2000`), so the calibration here is the calibration of the rule the runner uses. Every labelled query with at least one relevant document is used.

**Populations.** `ref_pool(pop, ref_i)` returns a reference window and the disjoint pool. For `ref_i = 0` it reproduces `run_dataset`'s split exactly (`np.random.default_rng(seed).permutation(n)`, first `W` = reference); other `ref_i` values draw further references from the same population so the across-reference variation can be measured. The runner keeps every test query that has qrels; this notebook keeps those with at least one positive judgment, and the cell prints how many test queries that drops (if zero, the positions are identical to the runner's).

**Tie-breaking.** The notebook breaks exact score ties by document index (stable sort); `pipeline.Retriever` uses FAISS `IndexFlatIP`, whose tie order can differ for the 29 queries tied at rank 10, so per-query nDCG can differ from the runner's clean baseline in the 4th decimal of the mean.

In [ ]:
def doc_text(title, text):
    return f"{(title or '').strip()} {(text or '').strip()}".strip()


def load_nfcorpus():
    c = NFCORPUS
    corpus = load_dataset(c["repo"], "corpus", split="corpus", revision=c["revision"])
    queries = load_dataset(c["repo"], "queries", split="queries", revision=c["revision"])
    qtext = {str(q["_id"]): q["text"] for q in queries}
    split_of, qrels = {}, {}
    for split in SPLITS:
        for row in load_dataset(c["qrels"], split=split, revision=c["qrels_revision"]):
            q = str(row["query-id"])
            if split_of.setdefault(q, split) == split:
                qrels.setdefault(q, {})[str(row["corpus-id"])] = int(row["score"])
    doc_ids = [str(d["_id"]) for d in corpus]
    docs = [doc_text(t, x) for t, x in zip(corpus["title"], corpus["text"])]
    return doc_ids, docs, {q: qtext[q] for q in qrels}, qrels, split_of


def ndcg_at_k(ranked, rels, k=K):
    dcg = sum(rels.get(d, 0) / np.log2(i + 2) for i, d in enumerate(ranked[:k]))
    ideal = sorted(rels.values(), reverse=True)[:k]
    idcg = sum(r / np.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def encode(enc, texts, batch_size=32):
    return np.ascontiguousarray(enc.encode(list(texts), batch_size=batch_size, normalize_embeddings=True,
                                           convert_to_numpy=True, show_progress_bar=False), dtype=np.float32)


def search(D, Q, k=K):
    S = Q @ D.T
    # Stable full sort: exact score ties (the corpus contains duplicate documents) break by document index,
    # so the top-k does not depend on which search routine is used.
    top = np.argsort(-S, axis=1, kind="stable")[:, :k]
    return top, np.take_along_axis(S, top, axis=1)


def paired_bootstrap(clean, damaged, n_boot=2000, seed=0):
    # identical to src/silent_rag/eval/ground_truth.py:paired_bootstrap
    rng = np.random.default_rng(seed)
    diff = clean - damaged
    boots = diff[rng.integers(0, len(diff), size=(n_boot, len(diff)))].mean(axis=1)
    return float(diff.mean()), float((boots <= 0).mean())


doc_ids, docs, queries, qrels, split_of = load_nfcorpus()
qids = [q for q in queries if any(r > 0 for r in qrels[q].values())]
rel = [{d for d, v in qrels[q].items() if v > 0} for q in qids]
N = len(qids)


def run_model(label):
    spec = MODELS[label]
    enc = SentenceTransformer(spec["repo"], revision=spec["revision"], device="cpu")
    enc.max_seq_length = MAX_SEQ
    D = encode(enc, docs)
    Q = encode(enc, [queries[q] for q in qids])
    top, scores = search(D, Q)
    ids = [[doc_ids[j] for j in row] for row in top]
    ndcg = np.array([ndcg_at_k(r, qrels[q]) for r, q in zip(ids, qids)])
    recall = np.array([len(r & set(i)) / len(r) for r, i in zip(rel, ids)])
    return {"D": D, "Q": Q, "scores": scores, "ndcg": ndcg, "recall": recall}


r0 = run_model("multi-qa-MiniLM")
alt = run_model("all-MiniLM")
sc = r0["scores"]
feat = pd.DataFrame({"top1": sc[:, 0], "top10_mean": sc.mean(1), "gap": sc[:, 0] - sc[:, 1]})
assert len(feat) == len(r0["ndcg"]) == len(alt["ndcg"]) == N   # every array is aligned by query position
print(N, "queries | mean nDCG@10", round(float(r0["ndcg"].mean()), 4), "| failure rate (recall@10 = 0)", round(float((r0["recall"] == 0).mean()), 3))

# Query populations. "runner" mirrors src/silent_rag/run.py: data.load(split="test") keeps the test queries in qrels
# order (the same relative order as here), then run_dataset permutes them with default_rng(seed): first W = reference.
test_pos = np.array([i for i, q in enumerate(qids) if split_of[q] == "test"])
n_test_with_qrels = sum(split_of[q] == "test" for q in queries)
POPS = {"runner": test_pos, "all": np.arange(N)}


def ref_pool(pop, ref_i, seed=SEED):
    base = POPS[pop]
    perm = np.random.default_rng(seed if ref_i == 0 else [seed, ref_i]).permutation(len(base))
    return base[perm[:W]], base[perm[W:]]


rows = []
for pop, base in POPS.items():
    n_pool = len(base) - W
    rows.append({"population": pop, "queries": len(base), "reference": W, "pool": n_pool,
                 "fpc_sd_factor": np.sqrt((n_pool - W) / (n_pool - 1)), "expected_overlap_between_two_windows": W * W / n_pool,
                 "mean_ndcg": r0["ndcg"][base].mean()})
print("test queries with qrels:", n_test_with_qrels, "| with a positive judgment (used here):", len(test_pos))
pd.DataFrame(rows).round(4)

## 1. Query-level AUROC for failing queries

A query fails when recall@10 = 0. AUROC is for the label-free score predicting failure (a lower score means more likely to fail, so the score is negated). A cross-validated logistic regression on the three scores shows what combining them buys. AUROC = 0.5 is chance. Shown for both populations.

**Why this test**
- **Estimand:** the probability that a random failing query has a lower top-1 (or other) score than a random succeeding one.
- **Unit of independence:** the query (bootstrap over queries). This is optimistic: queries share topics and relevant documents (notebook 04 shows the CI for a correlation roughly doubles under topic clustering), so treat the CI as a lower bound on uncertainty.
- **Assumptions:** none on the score distribution (AUROC is rank-based); the percentile bootstrap assumes the AUROC's sampling distribution is roughly symmetric, which holds away from 0 and 1.
- **Why not the obvious alternative:** a Mann-Whitney p-value only says AUROC differs from 0.5, which is not in doubt; the size of the separation is the quantity of interest.
- **Selection caveat:** top-1 was chosen as the detector signal in notebook 04 using its correlation with nDCG on these same queries, so every top-1 number here is in-sample.

In [ ]:
def auc_ci(y, s, B=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    a = roc_auc_score(y, s)
    boot = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        boot.append(roc_auc_score(y[i], s[i]))
    return a, *np.percentile(boot, [2.5, 97.5])


y_fail_all = (r0["recall"] == 0).astype(int)
rows = []
for pop, base in POPS.items():
    y_fail = y_fail_all[base]
    for f in feat:
        a, lo, hi = auc_ci(y_fail, -feat[f].to_numpy()[base])
        rows.append({"population": pop, "signal": f, "auroc_for_failure": a, "ci_lo": lo, "ci_hi": hi, "n": len(y_fail), "failure_rate": y_fail.mean()})
    p = cross_val_predict(LogisticRegression(max_iter=1000), feat.to_numpy()[base], y_fail,
                          cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="decision_function")
    a, lo, hi = auc_ci(y_fail, p)
    rows.append({"population": pop, "signal": "logistic(top1, top10_mean, gap), 5-fold CV", "auroc_for_failure": a, "ci_lo": lo, "ci_hi": hi,
                 "n": len(y_fail), "failure_rate": y_fail.mean()})
pd.DataFrame(rows).round(3)

## 2. Noise floor of window-mean nDCG@10

**Model.** A window is a simple random sample of `w` queries without replacement from the **pool** (the population minus the reference, as in the runner). Its mean has variance `sigma^2 / w * (P - w) / (P - 1)` (finite-population correction), where `P` is the pool size and `sigma` the SD of per-query nDCG over the pool. **Assumption tested:** the empirical SD of 3,000 random windows matches this formula, and window means are close to normal (skewness and a Shapiro-Wilk test on 500 windows), because z-thresholds on window means rely on it. nDCG is bounded and zero-inflated, so normality is not guaranteed at small windows.

**Why this test**
- **Estimand:** the SD of a clean window's mean nDCG@10, i.e. the noise any window-level rule must see through.
- **Unit of independence:** the window, drawn independently from a fixed pool (windows from the same pool share queries; that is what the finite-population term accounts for).
- **Assumptions:** simple random sampling. Real traffic is topically clustered, which this does not model (see Findings).
- **Why not the obvious alternative:** an iid formula `sigma / sqrt(w)` overstates the noise when the pool is small: with the runner's 223-query pool, a 100-query window is 45% of the pool.
- **Resolution:** with 3,000 windows the empirical SD has a relative Monte-Carlo error of about 1.3%; Shapiro-Wilk on 500 windows only detects clear departures from normality.

In [ ]:
rows = []
for pop in POPS:
    _, pool = ref_pool(pop, 0)
    nd = r0["ndcg"][pool]
    P = len(nd)
    sigma = nd.std(ddof=0)
    rng = np.random.default_rng([SEED, 2])
    for w in (25, 50, 100, 200, 400):
        if w >= P:
            continue
        means = np.array([nd[rng.choice(P, w, replace=False)].mean() for _ in range(3000)])
        theory = sigma / np.sqrt(w) * np.sqrt((P - w) / (P - 1))
        rows.append({"population": pop, "w": w, "pool": P, "empirical_sd": means.std(ddof=1), "theory_sd": theory,
                     "iid_sd_no_fpc": sigma / np.sqrt(w), "ratio": means.std(ddof=1) / theory,
                     "skew": stats.skew(means), "shapiro_p": stats.shapiro(means[:500]).pvalue,
                     "p2.5": np.percentile(means, 2.5), "p97.5": np.percentile(means, 97.5)})
pd.DataFrame(rows).round(4)

## 3. Calibration of the `harmful` rule at w = 100

The benchmark calls a condition harmful when `paired_bootstrap(clean, damaged)` has p < 0.05. If clean and damaged are exchangeable (no real damage), the false-positive rate should be about 5%. To test it with real per-query nDCG, a **sign-flip null** is built from the per-query differences `d_i` between the baseline and `all-MiniLM`: the "damaged" nDCG of a window is `clean_i - s_i * d_i` with random signs `s_i = +/-1`, which has a symmetric, zero-mean paired difference by construction but keeps the real, skewed, zero-inflated variance of nDCG. 1,000 windows of 100 from each population's pool, 2,000 bootstrap resamples each (the runner's setting).

**Why this test**
- **Estimand:** the false-positive rate of `harmful = p < 0.05` when there is no true drop.
- **Unit of independence:** the window; within a window, queries are resampled with replacement, as the rule itself does.
- **Assumptions:** the sign-flip null is valid when the paired difference is symmetric about zero under "no damage". Real injected damage is one-sided, so this checks the machinery, not the benchmark.
- **Why not the obvious alternative:** a paired t-test assumes a near-normal mean difference; with zero-inflated, bounded nDCG and 100 queries the bootstrap is the rule actually used, so its own calibration is what matters. Comparing the rule with itself on identical clean data is useless (all differences are exactly zero, so p = 1).
- **Resolution:** 1,000 windows give a binomial SE of about 0.7 percentage points at 5%; the Jeffreys interval is reported.

In [ ]:
d_model = r0["ndcg"] - alt["ndcg"]
print("per-query difference SD between two clean models:", round(float(d_model.std()), 4), "| mean:", round(float(d_model.mean()), 4))

rows, pv_store = [], {}
for pop in POPS:
    _, pool = ref_pool(pop, 0)
    rng = np.random.default_rng([SEED, 3])
    pvals = []
    for j in range(1000):
        idx = rng.choice(pool, W, replace=False)
        s = rng.choice([-1.0, 1.0], W)
        clean = r0["ndcg"][idx]
        pvals.append(paired_bootstrap(clean, clean - s * d_model[idx], n_boot=2000, seed=j)[1])
    pvals = np.array(pvals)
    k_ = int((pvals < 0.05).sum())
    lo, hi = stats.beta.ppf([0.025, 0.975], k_ + 0.5, len(pvals) - k_ + 0.5)
    rows.append({"population": pop, "windows": len(pvals), "false_positive_rate": k_ / len(pvals), "jeffreys_lo": lo, "jeffreys_hi": hi,
                 "uniformity_ks_p": stats.kstest(pvals, "uniform").pvalue})
    pv_store[pop] = pvals
display(pd.DataFrame(rows).round(4))

fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
for ax, (pop, pvals) in zip(axes, pv_store.items()):
    ax.hist(pvals, bins=20, color="lightgrey")
    ax.set_title(pop, fontsize=9)
    ax.set_xlabel("paired_bootstrap p under the sign-flip null")
axes[0].set_ylabel("windows")
fig.tight_layout()
plt.show()

## 4. Calibration of KS and z detectors against a fixed reference window

Detectors compare a scoring window with a **reference window** drawn once from the clean system. Two label-free window statistics are used: the two-sample **KS** test, and the **z-score of the window mean** against the reference, each on the top-1 score and on the top-1 gap. The reference is disjoint from every scoring window. The rate is measured over 200 reference draws per population, each scored on 300 clean windows, giving the *conditional* false-alarm rate given a reference (what a single benchmark run experiences).

**The z detector (corrected).** An earlier version divided by `sd_ref / sqrt(w)`, which ignores that the reference *mean* is itself estimated from 100 queries; with equal sizes the expected false-alarm rate at 1.96 is `P(|Z| > 1.96 / sqrt 2) = 16.6%`. The detector now uses the **Welch** standard error `sqrt(var_ref / n_ref + var_win / w)`, which counts both means' error and does not assume the damaged window has the reference's variance (damage changes it). The naive form is kept only as a labelled `z naive SE (bug)` row so the size of the error stays visible.

**Why this test**
- **Estimand:** the false-alarm rate of each rule on clean windows, conditional on the reference, and how much it varies across references.
- **Unit of independence:** the reference (200 draws); the 300 windows per reference estimate that reference's conditional rate.
- **Assumptions:** KS assumes continuous data; the top-1 gap has exact ties (gap = 0 for duplicate documents), which makes KS slightly conservative there. The z rule assumes the difference of means is near-normal (section 2 supports this at w = 100).
- **Why not the obvious alternative:** an unconditional rate (pooling all references) hides the fact that a single run is stuck with one reference; the across-reference spread is the practically relevant number.
- **Resolution:** each conditional rate has a binomial SE of about 1.3 points at 5% (300 windows).

In [ ]:
def window_stats(feat_ref, feat_win):
    """KS statistic, KS p, Welch z, and the old naive z (reference SD only, reference mean treated as known)."""
    ks = ks_2samp(feat_ref, feat_win)
    diff = feat_win.mean() - feat_ref.mean()
    se = np.sqrt(feat_ref.var(ddof=1) / len(feat_ref) + feat_win.var(ddof=1) / len(feat_win))
    z_naive = diff / (feat_ref.std(ddof=1) / np.sqrt(len(feat_win)) + 1e-12)
    return np.array([ks.statistic, ks.pvalue, diff / (se + 1e-12), z_naive])


cal_rows = []
for pop in POPS:
    for sig in ("top1", "gap"):
        x = feat[sig].to_numpy()
        rng = np.random.default_rng([SEED, 4])
        cond = {"KS p<0.05": [], "|z|>1.96 Welch SE": [], "|z|>1.96 naive SE (bug)": []}
        for ref_i in range(200):
            ref, pool = ref_pool(pop, ref_i)
            st = np.array([window_stats(x[ref], x[rng.choice(pool, W, replace=False)]) for _ in range(300)])
            cond["KS p<0.05"].append((st[:, 1] < 0.05).mean())
            cond["|z|>1.96 Welch SE"].append((np.abs(st[:, 2]) > 1.96).mean())
            cond["|z|>1.96 naive SE (bug)"].append((np.abs(st[:, 3]) > 1.96).mean())
        for rule, v in cond.items():
            v = np.array(v)
            cal_rows.append({"population": pop, "signal": sig, "rule": rule, "mean_false_alarm": v.mean(), "sd_across_references": v.std(ddof=1),
                             "p5": np.percentile(v, 5), "p95": np.percentile(v, 95), "max": v.max(), "runner_reference_0": v[0]})
pd.DataFrame(cal_rows).round(4)

## 5. Window-level tracking (hypothesis H3)

Partition the queries into disjoint windows of `w`, compute window-mean nDCG and window-mean signal, and correlate across windows (Spearman). The statistic is the mean of that correlation over **200 random partitions**. Windows from different partitions share queries, so they are not independent; inference comes from a **permutation null**: shuffle the signal across queries (breaking its link to nDCG) and compute **the same statistic** (again the mean over 200 partitions), 300 times. An earlier version averaged 300 partitions for the observed value but only 20 for each null value, so the null was wider than the null of the observed statistic (conservative p, inflated null SD); both now use 200. Window sizes 50, 100 and 200 show how the signal strengthens with aggregation.

**Population.** H3 is run on **all** labelled queries only: the runner's 323 test queries give three windows of 100, too few for a correlation across windows.

**H3 (confirmatory):** at w = 100 the window-mean top-1 score correlates positively with window-mean nDCG@10 (one-sided; in notebook 04, H1 and H2 are corrected together with Holm, H3 is tested here on its own).

**Why this test**
- **Estimand:** the expected Spearman correlation between window-mean signal and window-mean nDCG for a random partition into windows of `w`.
- **Unit of independence:** the query (the permutation shuffles queries' signal values).
- **Assumptions:** exchangeability of the signal across queries under the null of no association.
- **Why not the obvious alternative:** the p-value of a single partition's Spearman uses only `N / w` windows and depends on which partition is drawn; averaging over partitions removes that arbitrariness, and the permutation null keeps the test exact for the averaged statistic.
- **Resolution:** the smallest attainable p is 1/301 = 0.0033, reported as `p < 0.004` when the observed value exceeds every null value.

In [ ]:
def partition_corr(x, y, w, parts, rng):
    nw = len(x) // w
    out = np.empty(parts)
    for p_ in range(parts):
        idx = rng.permutation(len(x))[: nw * w].reshape(nw, w)
        out[p_] = stats.spearmanr(x[idx].mean(1), y[idx].mean(1))[0]
    return out


PARTS, NULL_REPS = 200, 300
y = r0["ndcg"]
rows, store = [], {}
for sig in ("top1", "top10_mean", "gap"):
    x = feat[sig].to_numpy()
    q_rho = stats.spearmanr(x, y)[0]
    for w in (50, 100, 200):
        rng = np.random.default_rng([SEED, 5])
        obs = partition_corr(x, y, w, PARTS, rng).mean()
        null = np.array([partition_corr(rng.permutation(x), y, w, PARTS, rng).mean() for _ in range(NULL_REPS)])  # same statistic
        idx = np.random.default_rng(SEED).permutation(len(y))[: (len(y) // w) * w].reshape(-1, w)
        slope = stats.linregress(x[idx].mean(1), y[idx].mean(1)).slope
        rows.append({"signal": sig, "w": w, "n_windows": len(y) // w, "query_level_rho": q_rho, "window_level_rho_mean": obs,
                     "null_mean": null.mean(), "null_sd": null.std(ddof=1), "null_max": null.max(),
                     "perm_p_one_sided": (1 + (null >= obs).sum()) / (1 + len(null)), "slope_ndcg_per_unit_signal": slope})
        store[(sig, w)] = (obs, null)
display(pd.DataFrame(rows).round(4))

obs, null = store[("top1", 100)]
fig, ax = plt.subplots(figsize=(4, 2.8))
ax.hist(null, bins=25, color="lightgrey")
ax.axvline(obs, color="tab:red")
ax.set_title("top-1 score, w=100 (all queries)", fontsize=9)
ax.set_xlabel("mean window-level Spearman")
fig.tight_layout()
plt.show()

## 6. Power and earliness under a synthetic degradation

**Synthetic degradation (not F1-F5).** In a damaged window, a fraction `s` (the severity) of the queries return **random documents** instead of their real results. For those queries the top-1 score becomes the best cosine to 10 random documents and nDCG@10 is recomputed on the random list. This gives a known, label-based effect to test the statistics against.

**Design (second-review changes, so it matches the runner).**
- *Paired across severities.* For each reference and trial `m`, the window, the order in which queries get damaged and the random documents are drawn once, from `default_rng([SEED, ref_i, m])`, and reused at every severity. Severity `s` damages the first `round(s * 100)` queries of that order, so damage is **nested**: a higher severity damages a superset of queries. The runner likewise scores the same window for every condition of trial `t`. An earlier version redrew windows and the affected set per severity and used one fixed set of random documents for the whole notebook, so severity curves mixed window noise into the severity effect and power was conditional on one random draw.
- *Severity grid* `0, 0.05, 0.1, 0.2, ..., 0.9`, covering the project's 0.1-0.9 range.
- *Harmful rule* with `n_boot = 2000` and `seed = m`, exactly as `run.py` calls it (`seed=trial`). At severity 0 the damaged window equals the clean one, so its rate is 0 by construction; its false-positive rate is the sign-flip result in section 3.
- *30 references* per population; reference 0 of `runner` is the runner's actual reference.

**Rules scored (on the top-1 score).**
- **Nominal:** KS p < 0.05 and Welch |z| > 1.96, what a user would pick without calibration; the naive-SE z is kept as a labelled bug row.
- **Oracle-calibrated:** the threshold is the 95th percentile of the statistic on 300 clean windows drawn from the pool. This is an **upper bound**: a `Detector` only sees `fit(reference)`, so it cannot draw clean windows from the pool.
- **Reference-only calibration:** the threshold is the 95th percentile of the statistic between the reference and 300 bootstrap resamples of the reference itself, which a detector *can* compute inside `fit()`. Its severity-0 rate shows whether this is calibrated.

**Earliness** is the smallest severity `s` such that the detection rate is at least 80% at `s` **and at every higher severity** (per reference), so one noisy early crossing does not count.

**Why this test**
- **Estimand:** the detection probability of each rule at each severity, for a reference drawn as the runner draws it, and its spread across references.
- **Unit of independence:** the reference (30 draws); 200 trials per reference estimate the conditional rate (binomial SE at most 3.5 points).
- **Assumptions:** none beyond the synthetic damage model; detection rates are empirical.
- **Why not the obvious alternative:** an analytic power formula needs normal, symmetric differences, which zero-inflated nDCG with all-or-nothing damage does not give (section 7 compares them).
- **Interpretation of the error bars:** markers are the mean over references; bars are the 5th-95th percentiles across references, the variation a single run is exposed to. No lines connect the markers, because nothing is known between grid points.

In [ ]:
D0, Q0 = r0["D"], r0["Q"]
clean_top1, clean_nd = feat["top1"].to_numpy(), r0["ndcg"]


def random_lists(win, rng, n_docs=10):
    """Top-1 score and nDCG@10 if each query in `win` returned 10 random documents (fresh draw per call)."""
    t1, nd_r = np.empty(len(win)), np.empty(len(win))
    for j, i in enumerate(win):
        pick = rng.choice(len(D0), n_docs, replace=False)
        t1[j] = (D0[pick] @ Q0[i]).max()
        nd_r[j] = ndcg_at_k([doc_ids[p] for p in pick], qrels[qids[i]])
    return t1, nd_r


def p95_thresholds(st):
    return np.percentile(st[:, 0], 95), np.percentile(np.abs(st[:, 2]), 95)


sev_grid = (0.0, 0.05) + tuple(np.round(np.arange(0.1, 1.0, 0.1), 1))
N_REF, M_DAM, M_CLEAN, M_REFBOOT = 30, 200, 300, 300
RULES = ["harmful_rate", "ks_nominal", "z_nominal", "z_naive_bug", "ks_oracle", "z_oracle", "ks_refonly", "z_refonly"]

rows, trials_store = [], {}
for pop in POPS:
    for ref_i in range(N_REF):
        ref, pool = ref_pool(pop, ref_i)
        xr = clean_top1[ref]
        rng = np.random.default_rng([SEED, 6, ref_i])
        cl = np.array([window_stats(xr, clean_top1[rng.choice(pool, W, replace=False)]) for _ in range(M_CLEAN)])
        rb = np.array([window_stats(xr, xr[rng.integers(0, W, W)]) for _ in range(M_REFBOOT)])
        (ks_or, z_or), (ks_rf, z_rf) = p95_thresholds(cl), p95_thresholds(rb)
        trials = []
        for m in range(M_DAM):
            rt = np.random.default_rng([SEED, ref_i, m])
            win = rt.choice(pool, W, replace=False)
            trials.append((win, rt.permutation(W)) + random_lists(win, rt))
        if ref_i == 0:
            trials_store[pop] = trials
        for s in sev_grid:
            k = int(round(s * W))
            hits = dict.fromkeys(RULES, 0)
            dam = []
            for m, (win, order, rt1, rnd) in enumerate(trials):
                t1, nd_w = clean_top1[win].copy(), clean_nd[win].copy()
                t1[order[:k]], nd_w[order[:k]] = rt1[order[:k]], rnd[order[:k]]
                _, p_h = paired_bootstrap(clean_nd[win], nd_w, n_boot=2000, seed=m)   # as run.py: seed=trial
                ks_stat, ks_p, z, z_naive = window_stats(xr, t1)
                for rule, fired in (("harmful_rate", p_h < 0.05), ("ks_nominal", ks_p < 0.05), ("z_nominal", abs(z) > 1.96),
                                    ("z_naive_bug", abs(z_naive) > 1.96), ("ks_oracle", ks_stat > ks_or), ("z_oracle", abs(z) > z_or),
                                    ("ks_refonly", ks_stat > ks_rf), ("z_refonly", abs(z) > z_rf)):
                    hits[rule] += fired
                dam.append((ks_stat, abs(z)))
            dam = np.array(dam)
            yy = np.r_[np.ones(M_DAM), np.zeros(M_CLEAN)]
            aucs = [roc_auc_score(yy, np.r_[dam[:, 0], cl[:, 0]]), roc_auc_score(yy, np.r_[dam[:, 1], np.abs(cl[:, 2])])] if s > 0 else [np.nan, np.nan]
            rows.append({"population": pop, "reference": ref_i, "severity": s, **{r_: v / M_DAM for r_, v in hits.items()},
                         "auroc_ks": aucs[0], "auroc_z": aucs[1]})
by_ref = pd.DataFrame(rows)
cols = RULES + ["auroc_ks", "auroc_z"]
g = by_ref.groupby(["population", "severity"])[cols]
power = g.mean().round(3)
spread = (g.quantile(0.95) - g.quantile(0.05)).round(3)
print("mean detection rate over 30 references")
display(power)
print("width of the 5th-95th percentile range across references")
display(spread)
print("runner's own reference (reference 0, population 'runner')")
by_ref[(by_ref["population"] == "runner") & (by_ref["reference"] == 0)].set_index("severity")[cols].round(3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharey=True, sharex=True)
panels = (("nominal thresholds", ["harmful_rate", "ks_nominal", "z_nominal"]),
          ("calibrated: oracle (solid) / reference-only (open)", ["harmful_rate", "ks_oracle", "z_oracle", "ks_refonly", "z_refonly"]))
offsets = dict(zip(RULES, np.linspace(-0.012, 0.012, len(RULES))))
for row_axes, pop in zip(axes, POPS):
    d = by_ref[by_ref["population"] == pop]
    gm = d.groupby("severity")[RULES]
    mean, lo, hi = gm.mean(), gm.quantile(0.05), gm.quantile(0.95)
    for ax, (title, series) in zip(row_axes, panels):
        for col in series:
            xs = mean.index + offsets[col]
            yerr = np.clip([mean[col] - lo[col], hi[col] - mean[col]], 0, None)   # clip float round-off at 0
            ax.errorbar(xs, mean[col], yerr=yerr, fmt="o", ms=4, capsize=2, ls="none",
                        mfc="none" if col.endswith("refonly") else None, label=col)
        ax.axhline(0.8, color="grey", ls=":")
        ax.axhline(0.05, color="grey", ls=":")
        ax.set_title(f"{pop}: {title}", fontsize=9)
for ax in axes[1]:
    ax.set_xlabel("fraction of queries degraded (markers only; no interpolation)")
for ax in axes[:, 0]:
    ax.set_ylabel("detection rate (mean, 5-95% over refs)")
axes[0, 0].legend(fontsize=7)
axes[0, 1].legend(fontsize=7)
fig.tight_layout()
plt.show()


def earliest_sustained(d, col, level=0.8):
    d = d[d["severity"] > 0].sort_values("severity")
    ok = (d[col].to_numpy() >= level)[::-1].cumprod()[::-1].astype(bool)   # True where this and every higher severity reach the level
    return float(d["severity"].to_numpy()[ok][0]) if ok.any() else np.inf


rows = []
for pop in POPS:
    for col in RULES:
        e = np.array([earliest_sustained(d, col) for _, d in by_ref[by_ref["population"] == pop].groupby("reference")])
        rows.append({"population": pop, "rule": col, "earliest_80pct_min": e.min(), "earliest_80pct_median": np.median(e),
                     "earliest_80pct_max": e.max(), "refs_never_reaching_80pct": int(np.isinf(e).sum())})
pd.DataFrame(rows)

## 6b. Does the harmful call depend on the bootstrap seed?

`harmful = p < 0.05` is a Monte-Carlo decision: `p` is estimated from `n_boot` resamples, so a window whose true bootstrap p is near 0.05 can be called harmful under one seed and not under another. The runner's `trials` are reproducible only because the seed is fixed (`seed=trial`). This section takes the runner population's reference-0 trials at severities 0.05 and 0.1 (where windows sit near the threshold), recomputes the call under 30 bootstrap seeds, and reports the share of windows whose call is **not unanimous** across seeds, for `n_boot` = 500 (the earlier power section's setting), 2,000 (the runner's) and 10,000.

**Why this test**
- **Estimand:** the probability that a window's harmful call flips with the bootstrap seed alone.
- **Unit of independence:** the window (100 per severity).
- **Assumptions:** none; this measures Monte-Carlo error directly.
- **Why not the obvious alternative:** the analytic SE of `p` (`sqrt(p(1 - p) / n_boot)`, about 0.005 at `n_boot = 2000`) says how far `p` moves, not how many benchmark rows change; counting flips does.

In [ ]:
rows = []
trials = trials_store["runner"][:100]
for s in (0.05, 0.1):
    k = int(round(s * W))
    for nb in (500, 2000, 10000):
        flips, base_rate = 0, []
        for win, order, rt1, rnd in trials:
            nd_w = clean_nd[win].copy()
            nd_w[order[:k]] = rnd[order[:k]]
            calls = np.array([paired_bootstrap(clean_nd[win], nd_w, n_boot=nb, seed=b)[1] < 0.05 for b in range(30)])
            flips += calls.min() != calls.max()
            base_rate.append(calls.mean())
        rows.append({"population": "runner", "severity": s, "n_boot": nb, "windows": len(trials),
                     "share_windows_call_depends_on_seed": flips / len(trials), "mean_harmful_rate": np.mean(base_rate)})
pd.DataFrame(rows).round(3)

In [ ]:
# Why more resamples do not help: with sparse damage the bootstrap p sits on a lattice.
# Only damaged queries whose clean nDCG was > 0 change (a failing query has nothing to lose). If j such queries
# have a positive drop and none a negative one, a resample's mean drop is 0 exactly when it misses all j of them,
# so p is about (1 - j/W)^W. For j = 3 that is 0.0476, just under 0.05: the call is then decided by Monte-Carlo noise.
lattice = pd.DataFrame({"effective_damaged_queries_j": range(1, 7), "approx_p": [(1 - j / W) ** W for j in range(1, 7)]})
display(lattice.round(4))

rows = []
k = int(round(0.05 * W))
for win, order, rt1, rnd in trials:
    nd_w = clean_nd[win].copy()
    nd_w[order[:k]] = rnd[order[:k]]
    d = clean_nd[win] - nd_w
    calls = np.array([paired_bootstrap(clean_nd[win], nd_w, n_boot=2000, seed=b)[1] < 0.05 for b in range(30)])
    rows.append({"j_positive_drop": int((d > 0).sum()), "any_negative_drop": bool((d < 0).any()),
                 "call_depends_on_seed": calls.min() != calls.max(), "harmful_share_over_seeds": calls.mean()})
lat = pd.DataFrame(rows)
lat.groupby("j_positive_drop").agg(windows=("call_depends_on_seed", "size"), share_seed_dependent=("call_depends_on_seed", "mean"),
                                   mean_harmful_share=("harmful_share_over_seeds", "mean"), any_negative=("any_negative_drop", "mean")).round(3)

## 7. Minimum detectable nDCG drop

For a paired window test at `alpha = 0.05` (one-sided) and 80% power, the minimum detectable mean drop is `(z_0.95 + z_0.80) * sd_d / sqrt(w)`, where `sd_d` is the SD of the per-query nDCG difference between clean and damaged. The size of `sd_d` depends on the failure, which is unknown until injection, so the first table is a sensitivity grid. (For the runner's small pool a finite-population factor would shrink this slightly; the grid ignores it and is conservative.)

The second table uses the `sd_d` measured under the synthetic degradation (averaged over 5 draws of random documents for every query in the pool) and compares the analytic MDE with **simulated** harmful-rule power (`n_boot = 2000`, as in the runner), for both populations. The degradation can assign damage two ways, and they are not interchangeable: **fixed fraction** (exactly `s * 100` queries per window, the power table in section 6) and **Bernoulli** (each query damaged independently with probability `s`, which adds variation in how many queries are hit, and is what the analytic `sd_d` describes). Both simulated powers are shown so the analytic and simulated numbers are compared like for like.

**Why this test**
- **Estimand:** the smallest mean nDCG drop the harmful rule detects 80% of the time at w = 100.
- **Unit of independence:** the query within a window.
- **Assumptions:** the analytic formula assumes a near-normal mean difference; nDCG differences under all-or-nothing damage are skewed, so the simulation is the reference and the formula is a sanity check.
- **Why not the obvious alternative:** simulation alone gives no rule of thumb for an unknown failure; the formula does, once its accuracy has been checked here.

In [ ]:
z_ = stats.norm.ppf(0.95) + stats.norm.ppf(0.80)
grid = pd.DataFrame({"sd_d": [0.1, 0.2, 0.3, 0.4]})
for w in (50, 100, 200, 400):
    grid[f"MDE_w{w}"] = z_ * grid["sd_d"] / np.sqrt(w)
display(grid.round(3))

rows = []
for pop in POPS:
    _, pool = ref_pool(pop, 0)
    rng = np.random.default_rng([SEED, 7])
    rnd_draws = np.array([random_lists(pool, rng)[1] for _ in range(5)])        # (5, pool) nDCG under random documents
    for s in (0.05, 0.1, 0.2, 0.3, 0.5):
        diffs = [clean_nd[pool] - np.where(rng.random(len(pool)) < s, rd, clean_nd[pool]) for rd in rnd_draws]
        mean_drop, sd_d = np.mean([d.mean() for d in diffs]), np.mean([d.std() for d in diffs])
        hits = 0
        for m in range(300):
            rt = np.random.default_rng([SEED, 7, m])
            win = rt.choice(pool, W, replace=False)
            _, rnd = random_lists(win, rt)
            a = rt.random(W) < s                                   # Bernoulli: each query damaged with probability s
            hits += paired_bootstrap(clean_nd[win], np.where(a, rnd, clean_nd[win]), n_boot=2000, seed=m)[1] < 0.05
        rows.append({"population": pop, "severity": s, "true_mean_drop": mean_drop, "sd_d_bernoulli": sd_d,
                     "analytic_MDE_w100": z_ * sd_d / np.sqrt(W),
                     "analytic_power_w100": float(stats.norm.cdf(mean_drop / (sd_d / np.sqrt(W)) - stats.norm.ppf(0.95))),
                     "simulated_power_bernoulli": hits / 300,
                     "simulated_power_fixed_fraction": float(by_ref[(by_ref["population"] == pop) & (by_ref["severity"] == s)]["harmful_rate"].mean())})
pd.DataFrame(rows).round(3)

In [ ]:
import datasets, sentence_transformers, sklearn, scipy, torch
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
       "sklearn": sklearn.__version__, "torch": torch.__version__, "sentence_transformers": sentence_transformers.__version__,
       "datasets": datasets.__version__, "seed": SEED, "device": "cpu", "window_size": W,
       "synthetic_degradation": "random retrieval for a fraction of queries"})

## Reproducibility check against the original analysis

The original analysis saved its results as tables under `results/tables/`. This notebook recomputes everything from scratch and was compared with those tables. Where a value differs, the cause was traced and, where it was a flaw in the method rather than a rounding effect, fixed here. **Two sources of tiny differences apply to every notebook:** (1) the corpus contains 39 pairs of identical document vectors, so 29 queries have an exact score tie at rank 10/11, and which document is kept depends on the search routine (mean nDCG@10 ranges 0.29581 to 0.29590 across routines). This notebook breaks ties by document index (stable sort), which makes the top-k deterministic. (2) The package versions differ from those that wrote the saved tables, so differences in the third decimal are expected.

The "all" column below is the like-for-like comparison with the original (all 3,237 queries). The runner population is new.

| Quantity | Original | This notebook: all queries | This notebook: runner (323) | Status |
|---|---|---|---|---|
| Query-level AUROC for failure (top-1 / top-10 mean / gap / logistic) | 0.785 / 0.732 / 0.614 / 0.785 | 0.785 / 0.732 / 0.614 / 0.785 | 0.793 / 0.745 / 0.623 / 0.783 | identical; runner CIs about 3x wider |
| Window-mean SD at w = 100 (empirical / theory) | 0.0297 / 0.0297 | 0.0294 / 0.0295 (pool of 3,137) | 0.0223 / 0.0226 (pool of 223) | now from the pool, not the whole set |
| Harmful-rule false-positive rate, sign-flip null | 0.053 | 0.050 (0.038 to 0.065) | **0.071 (0.056 to 0.088)** | **new: above 5% for the runner** |
| KS false-alarm rate, top-1 (mean over 200 references) | 0.0369 | 0.0423 | 0.0285 | references now drawn by `ref_pool` |
| z false-alarm rate, top-1: naive SE / Welch SE | 0.1698 / (0.056 corrected) | 0.181 / 0.056 | 0.164 / 0.051 | Welch is now the detector |
| Window-level rho(top-1, nDCG) at w = 100 | 0.4383 (null SD 0.04, from 20 partitions) | 0.463 (null SD 0.020, 200 partitions) | not computed (3 windows) | **matched null** |
| Earliness at 80%, median (harmful / KS calibrated / z calibrated) | 0.1 / 0.3 / 0.2 | 0.1 / 0.2 / 0.2 | 0.1 / 0.2 / 0.2 | sustained definition, finer grid |
| Harmful-rule power at severity 0.05, fixed fraction | 0.650 (n_boot 500) | 0.671 (n_boot 2000) | 0.670 | runner's n_boot |

**Second-review changes (why the numbers moved).**
1. *Population.* `src/silent_rag/data.py` loads only the test split, so the runner calibrates and scores on 323 queries: a 100-query reference and a 223-query pool. All 323 test queries have a positive judgment, so `ref_pool("runner", 0)` reproduces the runner's reference and pool exactly. With a pool of 223, a 100-query window is 45% of the pool. The window SD is therefore 0.022, not 0.030, and two trial windows share about 45 queries. Every window-level section now reports both populations, with the runner as primary.
2. *Harmful rule settings.* The power section used `n_boot = 500`; the runner uses 2,000, which this notebook now uses everywhere. On the runner population the sign-flip false-positive rate is 7.1% (Jeffreys 5.6 to 8.8%), against 5.0% on all queries. The 1,000 sign-flip windows are drawn from a 223-query pool and overlap heavily, so the Jeffreys interval, which assumes independent windows, is too narrow. The rule is somewhat liberal for this population, not shown to be badly miscalibrated.
3. *z detector.* The naive standard error is kept only as a labelled bug row. The Welch form averages 5.1% (runner) and 5.6% (all) false alarms.
4. *H3 null.* The observed value used 300 partitions and each null value 20, so the null was too wide (old null SD 0.04). Both now use 200: null SD 0.020, observed 0.463, p = 0.0033, the floor of 300 permutations.
5. *Severity design.* Windows, damage order and random documents are now drawn once per (reference, trial) and shared across severities, with nested damage and a grid up to 0.9. Earliness requires the rate to stay at or above 80% at every higher severity. Medians are unchanged or one grid step earlier. The ranges are wide: calibrated KS needs 0.2 to 0.4 across references.
6. *New checks:* reference-only calibration (achievable by a detector) next to the oracle calibration; the bootstrap-seed flip rate and its lattice explanation (section 6b).

## Findings
Numbers are from a full from-scratch run (seed 0, NFCorpus, window size 100). **Runner** = the 323 test queries the runner actually uses (reference 100, pool 223); **all** = all 3,237 labelled queries (sensitivity).

- **Window noise matches the finite-population formula** in both populations (ratio 0.98 to 1.02). At w = 100 the window SD is **0.022 (runner)** and 0.029 (all). The iid formula without the correction would say 0.030 for the runner, overstating its noise by a third. Window means are close to normal at w ≤ 100 (Shapiro p 0.37 to 0.88). At w = 200 the runner pool is nearly exhausted, and normality fails (p = 0.003).
- **The harmful rule is roughly, not exactly, calibrated.** Under a symmetric sign-flip null its false-positive rate is **7.1% (5.6 to 8.8%) for the runner** and 5.0% (3.8 to 6.5%) for all queries. The runner interval is too narrow because its windows overlap.
- **The harmful call is seed-dependent at low damage, and more resamples do not fix it.** At severity 0.05, 32% of the runner's windows get a different call under different bootstrap seeds, whether `n_boot` is 500, 2,000 or 10,000. The cause is a lattice:
  - With sparse damage, the bootstrap p is about `(1 - j/100)^100`, where j is the number of damaged queries whose nDCG actually dropped.
  - For j = 3 that is 0.0476, just under 0.05.
  - Windows with j = 3 are seed-dependent 97% of the time; windows with any other j, 0%.
  
  Real injectors at low severity also produce sparse differences, so harmful labels near this boundary are partly Monte-Carlo noise fixed by `seed=trial`.
- **Detector false alarms depend on the reference.** Mean false-alarm rates at a nominal 5%, top-1 score, runner population:

  | Rule | Mean false alarm | SD across references | Max |
  |---|---|---|---|
  | KS | 2.9% | 0.08 | 0.49 |
  | Welch z | 5.1% | 0.13 | 0.81 |
  | Naive z (bug) | 16.4% | – | – |

  On the runner's own reference, KS and Welch z raise no false alarms in 300 clean windows.
- **The top-1 score separates failing queries:** AUROC 0.79 (0.74 to 0.84) on the runner's queries and 0.785 (0.77 to 0.80) on all; combining three score signals adds nothing.
- **Window-level tracking (H3, all queries):** mean window-level Spearman between top-1 and nDCG is 0.46 at w = 100. The matched null is 0.00 ± 0.02, and p = 0.0033 is the floor of 300 permutations.
- **Power under the placeholder degradation** (30 references, mean detection rate):

  | Rule | Severity 0.05 | Severity 0.10 | Median 80% earliness |
  |---|---|---|---|
  | Harmful rule | 0.67 | 0.99 | 0.1 |
  | KS, oracle-calibrated | 0.09 | 0.24 | 0.2 (0.2 to 0.4 across references) |
  | Welch z, oracle-calibrated | 0.16 | 0.44 | 0.2 (0.1 to 0.4 across references) |
  | KS, nominal | – | – | 0.3 |
  | Welch z, nominal | – | – | 0.25 |

  The reference-only calibration, which a detector could compute inside `fit()`, is **not calibrated**: at severity 0 it alarms 22% (KS) and 10% (z) of the time on the runner population, and 37% / 22% on all queries.
- **Minimum detectable drop** for a 100-query window with a per-query SD of 0.2: about 0.05 nDCG. At severity 0.05 the analytic power (0.48) understates the simulated power (0.58 Bernoulli, 0.67 fixed fraction), because nDCG differences are skewed.

### What this supports
For the runner's own setting (323 NFCorpus test queries, a 100-query reference, windows of 100 from a 223-query pool, clean system):
- window-mean nDCG@10 has a known noise floor of about 0.022;
- the paired-bootstrap harmful rule is roughly calibrated (5 to 7% under a symmetric null);
- a Welch z or KS detector with a 100-query reference is much less sensitive than the ground-truth rule, and its false-alarm rate depends strongly on the reference.

### What this does not support
- How detectable **F1 to F5** are. The degradation here is a placeholder: damaged queries lose all of their nDCG, which is far easier to see than the subtle damage real injectors cause.
- That calibrated detector power is achievable: the oracle thresholds use clean pool windows the `Detector` interface does not see, and the achievable reference-only calibration over-alarms.
- That the runner's 20 trials are independent replicates: windows from a 223-query pool share about 45 queries per pair.
- That harmful labels near the p = 0.05 boundary are stable: at low damage they can flip with the bootstrap seed (section 6b).
- That the top-1 numbers are out-of-sample: top-1 was selected on these queries (notebook 04), and the hypotheses were not formed blind to the held-out datasets (README).
- That the noise floor applies to **production traffic**, which is topically clustered (topic grouping widens CIs about 1.3 to 2.4 times in notebook 04), or that thresholds transfer to another corpus, model or window size.